In [8]:
import requests
import pandas as pd
import sqlite3
import datetime

In [ ]:
def api_request(url, timeout_seconds=60):
    """Fetches API data with a defined timeout handling."""
    try:
        r = requests.get(url, timeout=timeout_seconds)
        if r.status_code == 200:
            return r.json()
        print(f"Error with API request, code: {r.status_code}")
    except requests.exceptions.Timeout:
        print(f"API request timed out after {timeout_seconds} seconds.")
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
    return None

In [3]:
def get_station_info(station_data, field_name='Street'):
    """Retrieves a specific metadata field for a station using its name."""
    try:
        stations_index = pd.read_csv("resources/indices_staions.csv").set_index('name')['index'].to_dict()
        idx = stations_index.get(field_name)
        return [item[idx] for item in station_data.values()] if idx is not None else []
    except FileNotFoundError:
        print("resources/indices_staions.csv not found.")
        return []

In [4]:
def filter_stations(stations, by_param="City", filter_val="Hamburg"):
    """Filters the station dictionary by a specified parameter."""
    try:
        stations_index = pd.read_csv("resources/indices_staions.csv").set_index('name')['index'].to_dict()
        idx = stations_index[by_param]
        return {k: v for k, v in stations.items() if v[idx] == filter_val}
    except KeyError as e:
        print(f"by_param: {by_param} is not a valid key. Error: {e}")
        return {}
    except FileNotFoundError:
        print("resources/indices_staions.csv not found.")
        return {}

In [5]:
def download_and_store_data(conn, stations, component, date_from, date_to):
    """Downloads environmental measurements and stores them in SQLite."""
    cursor = conn.cursor()
    base_url = "https://www.umweltbundesamt.de/api/air_data/v4/measures/json"
    
    for station_id in stations.keys():
        print(f"Fetching component {component} for station {station_id}...")
        url = (f"{base_url}?station={station_id}&component={component}"
               f"&date_from={date_from}&date_to={date_to}&time_from=1&time_to=24&lang=de")
               
        data = api_request(url)
        if not data or "data" not in data or str(station_id) not in data["data"]:
            continue
            
        records = []
        for timestamp, values in data["data"][str(station_id)].items():
            val = values[0] if isinstance(values, list) and values else values
            if val is not None:
                records.append((station_id, str(component), timestamp, val))
        
        cursor.executemany('''
            INSERT OR IGNORE INTO measurements (station_id, component_id, timestamp, value)
            VALUES (?, ?, ?, ?)
        ''', records)
        conn.commit()
        print(f"Stored {len(records)} entries for station {station_id}.")


In [ ]:
def init_db(db_name="capstone_air_data.db"):
    """Initializes the SQLite database to store measurement data."""
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS measurements (
            station_id TEXT,
            component_id TEXT,
            timestamp TEXT,
            value REAL,
            UNIQUE(station_id, component_id, timestamp)
        )
    ''')
    conn.commit()
    return conn

In [9]:
# 1. Fetch metadata
meta_url = "https://luftdaten.umweltbundesamt.de/api-proxy/meta/json?use=airquality&date_from=2025-06-01&date_to=2026-06-30&time_from=1&time_to=24&lang=de"
meta_data = api_request(meta_url)

component_select = 9
''' Other components are:
    {'1': ['1', 'PM10', 'PM₁₀', 'µg/m³', 'Feinstaub'],
    '2': ['2', 'CO', 'CO', 'mg/m³', 'Kohlenmonoxid'],
    '3': ['3', 'O3', 'O₃', 'µg/m³', 'Ozon'],
    '4': ['4', 'SO2', 'SO₂', 'µg/m³', 'Schwefeldioxid'],
    '5': ['5', 'NO2', 'NO₂', 'µg/m³', 'Stickstoffdioxid'],
    '6': ['6', 'PM10PB', 'Pb', 'µg/m³', 'Blei im Feinstaub'],
    '7': ['7', 'PM10BAP', 'BaP', 'ng/m³', 'Benzo(a)pyren im Feinstaub'],
    '8': ['8', 'CHB', 'C₆H₆', 'µg/m³', 'Benzol'],
    '9': ['9', 'PM2', 'PM₂,₅', 'µg/m³', 'Feinstaub'],
    '10': ['10', 'PM10AS', 'As', 'ng/m³', 'Arsen im Feinstaub'],
    '11': ['11', 'PM10CD', 'Cd', 'ng/m³', 'Cadmium im Feinstaub'],
    '12': ['12', 'PM10NI', 'Ni', 'ng/m³', 'Nickel im Feinstaub']}
    '''


if meta_data and "stations" in meta_data:
    # 2. Filter stations (e.g., Hamburg)
    hamburg_stations = filter_stations(meta_data["stations"], by_param="City", filter_val="Hamburg")
    
    # 3. Initialize Database
    str_db_filename = datetime.date.today().__str__() + component_select
    db_connection = init_db(db_name=str_today+"capstone_air.db")
    
    # 4. Download and store PM10 (component=1) for June 2025
    
    download_and_store_data(
        conn=db_connection, 
        stations=hamburg_stations, 
        component=component_select,
        date_from="2025-06-01", 
        date_to="2025-06-30"
    )
    
    db_connection.close()
    print("Data ingestion complete.")

KeyboardInterrupt: 